## Setup & Imports

# Image Preprocessing Pipeline for Diabetic Foot Ulcer Detection

## Complete Preprocessing Workflow (Chapter 3)

This notebook implements the full image preprocessing pipeline as described in the research methodology, following all steps outlined in Chapter 3.

## 1. Image Segmentation (HMRF-EM)

**Description:** Separate the foot from the background using Gaussian Mixture Model (GMM) with Hidden Markov Random Field (HMRF) and MAP (Maximum A Posteriori) estimation.

**Steps:**
- Convert RGB image to YDbDr color space
- Initialize 3 clusters (background, low-pressure, high-pressure) with K-means
- Run EM iterations with MAP labeling
- Apply segmentation mask to extract foot region

**Reference Code:** `preprocess_feet.py`

---

In [1]:
# Import Required Libraries
import os
import glob
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter, label, find_objects, binary_dilation
import cv2
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully")

# ============================================================
# 1. Color Space Conversion: RGB → YDbDr
# ============================================================

def rgb_to_ydbdr(image_rgb: np.ndarray) -> np.ndarray:
    """
    Convert an RGB image (H, W, 3) in [0, 255] to YDbDr.
    
    Y  =  0.299 R + 0.587 G + 0.114 B          (Luminance)
    Db = -0.450 R - 0.883 G + 1.333 B          (Chrominance blue)
    Dr = -1.333 R + 1.116 G + 0.217 B          (Chrominance red)
    
    Returns float64 array (H, W, 3).
    """
    img = image_rgb.astype(np.float64)
    R, G, B = img[..., 0], img[..., 1], img[..., 2]

    Y  =  0.299 * R + 0.587 * G + 0.114 * B
    Db = -0.450 * R - 0.883 * G + 1.333 * B
    Dr = -1.333 * R + 1.116 * G + 0.217 * B

    return np.stack([Y, Db, Dr], axis=-1)


# ============================================================
# 2. GMM Parameter Estimation (Expectation-Maximisation)
# ============================================================

class GaussianMixtureModel:
    """Diagonal-covariance GMM fitted with EM for D-dimensional data."""

    def __init__(self, K: int = 3, max_iter: int = 50, tol: float = 1e-4,
                 random_state: int = 42):
        self.K = K
        self.max_iter = max_iter
        self.tol = tol
        self.rng = np.random.RandomState(random_state)
        self.weights = None   # (K,)
        self.means = None     # (K, D)
        self.covs = None      # (K, D, D)

    def _init_params(self, X: np.ndarray):
        N, D = X.shape
        indices = self.rng.choice(N, self.K, replace=False)
        self.means = X[indices].copy()
        self.covs = np.array([np.eye(D) * np.var(X, axis=0) for _ in range(self.K)])
        self.weights = np.ones(self.K) / self.K

    @staticmethod
    def _log_gaussian(X: np.ndarray, mu: np.ndarray, cov: np.ndarray) -> np.ndarray:
        """Log pdf of multivariate Gaussian (N,) for (N, D) data."""
        D = X.shape[1]
        diff = X - mu
        sign, log_det = np.linalg.slogdet(cov)
        inv_cov = np.linalg.inv(cov)
        mahal = np.sum(diff @ inv_cov * diff, axis=1)
        return -0.5 * (D * np.log(2 * np.pi) + log_det + mahal)

    def fit(self, X: np.ndarray):
        """Run EM on (N, D) data."""
        N, D = X.shape
        self._init_params(X)
        prev_ll = -np.inf

        for iteration in range(self.max_iter):
            # E-step
            log_resp = np.zeros((N, self.K))
            for k in range(self.K):
                log_resp[:, k] = np.log(self.weights[k] + 1e-300) + \
                                 self._log_gaussian(X, self.means[k], self.covs[k])
            # Log-sum-exp trick
            log_resp_max = log_resp.max(axis=1, keepdims=True)
            log_norm = log_resp_max + np.log(
                np.sum(np.exp(log_resp - log_resp_max), axis=1, keepdims=True))
            log_resp -= log_norm
            resp = np.exp(log_resp)

            # M-step
            Nk = resp.sum(axis=0)
            self.weights = Nk / N
            for k in range(self.K):
                self.means[k] = (resp[:, k:k+1].T @ X) / Nk[k]
                diff = X - self.means[k]
                self.covs[k] = (diff.T * resp[:, k]) @ diff / Nk[k]
                self.covs[k] += np.eye(D) * 1e-6

            ll = np.sum(log_norm)
            if abs(ll - prev_ll) < self.tol:
                print(f"  GMM converged at iteration {iteration + 1}")
                break
            prev_ll = ll

        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Return (N, K) responsibility matrix."""
        N = X.shape[0]
        log_resp = np.zeros((N, self.K))
        for k in range(self.K):
            log_resp[:, k] = np.log(self.weights[k] + 1e-300) + \
                             self._log_gaussian(X, self.means[k], self.covs[k])
        log_resp_max = log_resp.max(axis=1, keepdims=True)
        log_norm = log_resp_max + np.log(
            np.sum(np.exp(log_resp - log_resp_max), axis=1, keepdims=True))
        log_resp -= log_norm
        return np.exp(log_resp)

        # ============================================================
# 3. HMRF-MAP Segmentation
# ============================================================

def _neighbourhood_label_counts(labels: np.ndarray, K: int) -> np.ndarray:
    """For each pixel, count how many of its 4-connected neighbours share each label."""
    H, W = labels.shape
    counts = np.zeros((H, W, K), dtype=np.float64)
    for k in range(K):
        binary = (labels == k).astype(np.float64)
        count_k = np.zeros_like(binary)
        count_k[1:, :]  += binary[:-1, :]   # top
        count_k[:-1, :] += binary[1:, :]    # bottom
        count_k[:, 1:]  += binary[:, :-1]   # left
        count_k[:, :-1] += binary[:, 1:]    # right
        counts[..., k] = count_k
    return counts


def hmrf_em_segmentation(image_ydbdr: np.ndarray, K: int = 3,
                         beta: float = 1.5, max_iter: int = 30,
                         tol: float = 1e-3) -> np.ndarray:
    """HMRF-EM segmentation with GMM."""
    H, W, D = image_ydbdr.shape
    X = image_ydbdr.reshape(-1, D)
    N = X.shape[0]

    print("[1/3] Fitting GMM (K={}) on YDbDr features ...".format(K))
    gmm = GaussianMixtureModel(K=K, max_iter=80)
    gmm.fit(X)

    resp = gmm.predict_proba(X)
    labels = resp.argmax(axis=1).reshape(H, W)

    print("[2/3] Running HMRF-EM MAP Labeling (beta={}) ...".format(beta))

    likelihood_energy = np.zeros((N, K))
    for k in range(K):
        likelihood_energy[:, k] = -GaussianMixtureModel._log_gaussian(
            X, gmm.means[k], gmm.covs[k])
    likelihood_energy = likelihood_energy.reshape(H, W, K)

    for it in range(max_iter):
        nbr_counts = _neighbourhood_label_counts(labels, K)
        prior_energy = beta * (4.0 - nbr_counts)

        total_energy = likelihood_energy + prior_energy

        new_labels = total_energy.argmin(axis=2)

        changed = np.mean(new_labels != labels)
        labels = new_labels
        if changed < tol:
            print(f"  ICM converged at iteration {it + 1} (changed = {changed:.6f})")
            break

    print("[3/3] Re-estimating GMM on final labels ...")
    for k in range(K):
        mask_k = (labels.ravel() == k)
        if mask_k.sum() < D + 1:
            continue
        gmm.means[k] = X[mask_k].mean(axis=0)
        gmm.covs[k] = np.cov(X[mask_k].T) + np.eye(D) * 1e-6

    return labels


# ============================================================
# 4. Identify Foot Label & Create Mask
# ============================================================

def identify_foot_label(labels: np.ndarray, image_ydbdr: np.ndarray) -> int:
    """Identify which cluster corresponds to the foot contact region."""
    K = labels.max() + 1
    scores = np.zeros(K)
    for k in range(K):
        mask_k = (labels == k)
        if mask_k.sum() == 0:
            continue
        mean_y  = image_ydbdr[mask_k, 0].mean()
        mean_db = np.abs(image_ydbdr[mask_k, 1]).mean()
        mean_dr = np.abs(image_ydbdr[mask_k, 2]).mean()
        chrom_mag = np.sqrt(mean_db**2 + mean_dr**2)
        scores[k] = mean_y * (1.0 + chrom_mag)

    foot_label = int(np.argmax(scores))
    scores_dict = {i: round(float(s), 1) for i, s in enumerate(scores)}
    print(f"  Cluster scores: {scores_dict}")
    return foot_label


def create_foot_mask(labels: np.ndarray, foot_label: int) -> np.ndarray:
    """Binary mask: 1 = foot, 0 = background."""
    return np.where(labels == foot_label, 1, 0).astype(np.uint8)


def get_pure_sole_image(image_rgb: np.ndarray, foot_mask: np.ndarray,
               background: str = "black") -> np.ndarray:
    """Apply mask to image."""
    mask_3ch = foot_mask[:, :, np.newaxis]
    if background == "white":
        bg = np.full_like(image_rgb, 255)
        masked = image_rgb * mask_3ch + bg * (1 - mask_3ch)
    else:
        masked = image_rgb * mask_3ch
    return masked.astype(np.uint8)

✓ All libraries imported successfully


## 2. Morphological Operations with Dilation

**Description:** Apply dilation along the vertical axis to bridge gaps between disconnected regions (e.g., toes and foot).

**Steps:**
- Use dynamic kernel (approximately 12% of image height for vertical, 5px for horizontal)
- Apply binary dilation to connect toes with the main foot area
- Output: Extended segmentation mask

---

In [2]:
# ============================================================
# SECTION 2: Morphological Operations with Dilation
# ============================================================

def apply_morphological_dilation(input_data, output_dir: str | None = None):
    """
    Apply morphological dilation to dilate foot region.
    
    Parameters
    ----------
    input_data : str or np.ndarray
        Path to segmented foot image OR segmented image array (H, W, 3)
    output_dir : str, optional
        Directory to save intermediate outputs (currently unused)
    
    Returns
    -------
    merged_mask : (H, W) binary array with dilated foot region
    image_array : (H, W, 3) original image array
    """
    print("=" * 60)
    print(f"  Morphological Dilation")
    print("=" * 60)

    # Load image if string, use directly if array
    if isinstance(input_data, str):
        img_pil = Image.open(input_data).convert("RGB")
        img_array = np.array(img_pil)
    else:
        img_array = input_data.copy()
    
    # Create binary mask from image
    intensity = img_array.sum(axis=2)
    binary_mask = (intensity > 0).astype(np.uint8)
    print(f"Binary mask created from image")
    
    # Dynamic kernel: 12% of image height for vertical, 5px for horizontal
    img_h, img_w = img_array.shape[:2]
    dilate_h = max(10, int(img_h * 0.12))
    struct_dilate = np.ones((dilate_h, 5), dtype=bool)
    print(f"Dilation kernel size: ({dilate_h}, 5)")
    
    # Apply binary dilation
    merged_mask = binary_dilation(binary_mask, structure=struct_dilate)
    print(f"Dilation applied - bridges gaps between disconnected regions (e.g., toes)")
    print("=" * 60)
    
    return merged_mask, img_array

## 3. Left-Right Separation & Square Crop

**Description:** Separate both footprints and create square images with consistent dimensions.

**Steps:**
- Apply Connected Component Analysis to find the 2 largest blobs
- Extract bounding boxes for left and right footprints
- Crop each foot tightly (from heel to toe)
- Pad shorter dimension with black pixels to create square images

---

In [3]:
# ============================================================
# SECTION 3: Left-Right Separation & Square Crop
# ============================================================

def pad_to_square(img_array: np.ndarray) -> np.ndarray:
    """Pad an image with black pixels to make it a perfect square."""
    h, w = img_array.shape[:2]
    size = max(h, w)
    
    pad_h_top = (size - h) // 2
    pad_h_bottom = size - h - pad_h_top
    pad_w_left = (size - w) // 2
    pad_w_right = size - w - pad_w_left
    
    padded = np.pad(img_array, ((pad_h_top, pad_h_bottom), 
                                (pad_w_left, pad_w_right), 
                                (0, 0)), 
                    mode='constant', constant_values=0)
    return padded


def separate_and_crop_feet(merged_mask: np.ndarray, img_array: np.ndarray, output_dir: str | None = None):
    """
    Separate left and right footprints and create square images.
    
    Parameters
    ----------
    merged_mask : (H, W) binary array with dilated foot region
    img_array : (H, W, 3) original image array
    output_dir : str, optional
        Directory to save intermediate visualizations
    
    Returns
    -------
    foot_left_img : (H, H, 3) left foot square image
    foot_right_img : (H, H, 3) right foot square image
    """
    print("=" * 60)
    print(f"  Left-Right Separation & Square Crop")
    print("=" * 60)
    
    # Connected Component Labeling (8-connected)
    structure = np.ones((3, 3), dtype=np.int32)
    labeled_mask, num_features = label(merged_mask, structure=structure)
    
    if num_features < 2:
        print("Error: Could not find at least 2 separate feet in the image.")
        return None, None
    
    print(f"Found {num_features} connected components")
    
    # Find the 2 Largest Components
    component_sizes = np.bincount(labeled_mask.ravel())
    component_sizes[0] = 0  # ignore background
    largest_labels = np.argsort(component_sizes)[-2:]
    print(f"2 largest components identified: labels {largest_labels}")
    
    # Extract Bounding Boxes
    slices = find_objects(labeled_mask)
    
    extracted_feet = []
    
    for lbl in largest_labels:
        bbox = slices[lbl - 1]
        y_slice, x_slice = bbox
        center_x = (x_slice.start + x_slice.stop) / 2.0
        
        # Tight crop
        cropped_img = img_array[y_slice, x_slice]
        
        # Pad to Square
        square_img = pad_to_square(cropped_img)
        
        extracted_feet.append({
            'center_x': center_x,
            'bbox': bbox,
            'square_img': square_img
        })
    
    # Sort by X coordinate (Leftmost = Left foot)
    extracted_feet.sort(key=lambda item: item['center_x'])
    foot_left_img  = extracted_feet[0]['square_img']
    foot_right_img = extracted_feet[1]['square_img']
    
    print(f"Left Foot:  {foot_left_img.shape[0]}x{foot_left_img.shape[1]}")
    print(f"Right Foot: {foot_right_img.shape[0]}x{foot_right_img.shape[1]}")
    print("=" * 60)
    
    return foot_left_img, foot_right_img

## 4. Grayscale Conversion & CLAHE Enhancement

**Description:** Convert to grayscale and enhance contrast using CLAHE to visualize pressure distribution better.

**Steps:**
- Convert RGB/segmented image to grayscale (focus on pressure intensity only)
- Apply Contrast Limited Adaptive Histogram Equalization (CLAHE)
  - Tile size: typically 8x8
  - Clip limit: prevents over-amplification of noise

**Formula:**
$$\text{Gray} = 0.299 \cdot R + 0.587 \cdot G + 0.114 \cdot B$$

---

In [ ]:
# ============================================================
# SECTION 4: Grayscale Conversion & CLAHE Enhancement
# ============================================================

def convert_to_grayscale(img_array: np.ndarray) -> np.ndarray:
    """
    Convert RGB image to grayscale using standard luminance formula.

    Formula: Gray = 0.299 * R + 0.587 * G + 0.114 * B
    """
    if len(img_array.shape) == 3 and img_array.shape[2] == 3:
        gray = 0.299 * img_array[..., 0] + 0.587 * img_array[..., 1] + 0.114 * img_array[..., 2]
        return gray.astype(np.uint8)
    else:
        return img_array


def apply_clahe(image_gray: np.ndarray, clip_limit: float = 3.5, tile_grid_size: tuple = (8, 8)) -> np.ndarray:
    """
    Apply Contrast Limited Adaptive Histogram Equalization (CLAHE).

    Parameters
    ----------
    image_gray : (H, W) uint8
    clip_limit : float
        Default 3.5, selected via CLAHE parameter tuning (Section 11).
    tile_grid_size : tuple
        Default (8, 8).

    Returns
    -------
    clahe_image : (H, W) uint8
    """
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    return clahe.apply(image_gray)

## 5. Image Standardization

### 5.1 Three-Channel RGB Conversion

**Description:** Convert grayscale image to 3-channel RGB for compatibility with pre-trained CNN models (ImageNet).

**Steps:**
- Copy grayscale values to all three channels: R = G = B
- Result: (H, W, 3) tensor

**Why:** Pre-trained models (EfficientNetB0, ResNet50, ConvNeXt-Tiny) expect 3-channel input

---

In [5]:
# ============================================================
# SECTION 5.1: Three-Channel RGB Conversion
# ============================================================

def convert_to_3channel_rgb(image_gray: np.ndarray) -> np.ndarray:
    """
    Convert grayscale image to 3-channel RGB by copying values to all channels.
    
    Parameters
    ----------
    image_gray : (H, W) uint8 or float grayscale image
    
    Returns
    -------
    image_rgb : (H, W, 3) 3-channel image
    """
    if len(image_gray.shape) == 2:
        # Stack grayscale 3 times to create RGB
        image_rgb = np.stack([image_gray, image_gray, image_gray], axis=-1)
        return image_rgb
    else:
        return image_gray

### 5.2 Image Resizing

**Description:** Resize all images to a uniform size (224×224 pixels) using bilinear interpolation.

**Specifications:**
- Target size: 224 × 224 pixels
- Interpolation method: Bilinear (preserves smooth transitions between pressure levels)
- Reason: Standard input size for EfficientNetB0, ResNet50, ConvNeXt-Tiny architectures

---

In [6]:
# ============================================================
# SECTION 5.2: Image Resizing
# ============================================================

def resize_image(image: np.ndarray, target_size: tuple = (224, 224), interpolation: str = 'bilinear') -> np.ndarray:
    """
    Resize image to target size using bilinear interpolation.
    
    Parameters
    ----------
    image : (H, W) or (H, W, C) array
    target_size : (height, width) tuple
    interpolation : 'bilinear' (default) or 'nearest'
    
    Returns
    -------
    resized_image : Array resized to target_size
    """
    pil_image = Image.fromarray(image)
    
    if interpolation == 'bilinear':
        interp = Image.BILINEAR
    else:
        interp = Image.NEAREST
    
    resized_pil = pil_image.resize((target_size[1], target_size[0]), interp)
    resized_array = np.array(resized_pil)
    
    return resized_array

### 5.3 Pixel Intensity Scaling (÷255)

**Description:** Scale pixel values to the range [0, 1] by dividing by 255 for storage as float32.

**Formula:**
$$x_{\text{scaled}} = \frac{x}{255}$$

**Purpose:**
- Produces a compact, numerically stable [0, 1] representation for storage
- A `Rescaling(255.0)` layer inside the model converts values back to [0, 255] before backbone-specific `preprocess_input` is applied

**Note:** This is NOT per-image min-max normalization — all images share the same fixed divisor (255), which preserves relative pressure intensity across images.

---

In [ ]:
# ============================================================
# SECTION 5.3: Pixel Intensity Scaling (÷255)
# ============================================================

def scale_pixels(image: np.ndarray) -> np.ndarray:
    """
    Scale pixel values to [0, 1] by dividing by 255.

    Uses a fixed global divisor (not per-image min-max), preserving
    relative pressure intensity across images.

    Parameters
    ----------
    image : np.ndarray
        Input image (uint8 or float in [0, 255])

    Returns
    -------
    scaled_image : np.ndarray float32
        Pixel values in [0, 1]
    """
    return image.astype(np.float32) / 255.0

## 7. Complete Pipeline Integration

**Description:** Create a unified preprocessing function that applies all steps in the correct order.

**Input:** Raw footprint image (RGB, variable size)

**Output:** Preprocessed image ready for CNN model training (3-channel, 224×224, scaled [0,1])

**Flow:**
1. Segmentation (HMRF-EM) → Mask
2. Apply Mask + Dilation → Connected Components
3. L-R Separation + Square Crop → Two square images
4. Grayscale + CLAHE Enhancement → Enhanced grayscale
5. 3-Channel Conversion → RGB (3-channel)
6. Resizing (224×224) → Standard size
7. Pixel Intensity Scaling (÷255) → [0, 1]

---

In [ ]:
# ============================================================
# SECTION 7: Complete Pipeline Integration
# ============================================================

def preprocess_foot_image(image_path: str, output_dir: str = None, target_size: tuple = (224, 224), save_final: bool = False):
    """
    Complete preprocessing pipeline for foot images.

    Input: Raw footprint image (RGB, variable size)
    Output: Preprocessed image ready for CNN (3-channel, 224×224, scaled [0,1])

    Flow:
    1. Segmentation (HMRF-EM) → Mask
    2. Apply Dilation → Extended segmentation
    3. L-R Separation + Square Crop → Two square images
    4. Grayscale + CLAHE → Enhanced grayscale
    5. 3-Channel Conversion → RGB
    6. Resizing (224×224) → Standard size
    7. Pixel Intensity Scaling (÷255) → [0, 1]
    """
    print("=" * 80)
    print(f"  COMPLETE IMAGE PREPROCESSING PIPELINE")
    print("=" * 80)

    # Step 1: Load and Segment
    print("\n[Step 1] Image Segmentation (HMRF-EM)...")
    img_pil = Image.open(image_path).convert("RGB")
    image_rgb = np.array(img_pil)

    image_ydbdr = rgb_to_ydbdr(image_rgb)
    labels = hmrf_em_segmentation(image_ydbdr, K=3, beta=1.5)
    foot_label = identify_foot_label(labels, image_ydbdr)
    foot_mask = create_foot_mask(labels, foot_label)
    pure_sole = get_pure_sole_image(image_rgb, foot_mask, background="black")

    # Step 2: Morphological Dilation
    print("\n[Step 2] Morphological Dilation...")
    merged_mask, _ = apply_morphological_dilation(pure_sole, output_dir=output_dir)

    # Step 3: L-R Separation & Square Crop
    print("\n[Step 3] L-R Separation & Square Crop...")
    foot_left, foot_right = separate_and_crop_feet(merged_mask, pure_sole, output_dir=output_dir)

    if foot_left is None or foot_right is None:
        print("Error: Failed to separate feet")
        return None

    # Step 4: Grayscale + CLAHE
    print("\n[Step 4] Grayscale & CLAHE Enhancement...")
    left_gray = convert_to_grayscale(foot_left)
    right_gray = convert_to_grayscale(foot_right)
    left_clahe = apply_clahe(left_gray, clip_limit=3.5, tile_grid_size=(8, 8))
    right_clahe = apply_clahe(right_gray, clip_limit=3.5, tile_grid_size=(8, 8))

    # Step 5: 3-Channel Conversion
    print("\n[Step 5] 3-Channel RGB Conversion...")
    left_rgb = convert_to_3channel_rgb(left_clahe)
    right_rgb = convert_to_3channel_rgb(right_clahe)

    # Step 6: Resizing
    print(f"\n[Step 6] Resizing to {target_size}...")
    left_resized = resize_image(left_rgb, target_size=target_size)
    right_resized = resize_image(right_rgb, target_size=target_size)

    # Step 7: Pixel Intensity Scaling (÷255)
    print("\n[Step 7] Pixel Intensity Scaling (÷255) → [0, 1]...")
    left_scaled = scale_pixels(left_resized)
    right_scaled = scale_pixels(right_resized)

    result = {
        'left_foot': left_scaled,
        'right_foot': right_scaled,
    }

    # Save final outputs if requested
    if save_final:
        base_name = os.path.splitext(os.path.basename(image_path))[0]

        left_output_dir  = os.path.join(output_dir, "left_foot")
        right_output_dir = os.path.join(output_dir, "right_foot")
        os.makedirs(left_output_dir,  exist_ok=True)
        os.makedirs(right_output_dir, exist_ok=True)

        left_output_path  = os.path.join(left_output_dir,  f"{base_name}_L.npy")
        right_output_path = os.path.join(right_output_dir, f"{base_name}_R.npy")

        np.save(left_output_path,  left_scaled)
        np.save(right_output_path, right_scaled)

        result['left_foot_path']  = left_output_path
        result['right_foot_path'] = right_output_path

        print(f"\n✓ Saved final outputs:")
        print(f"  Left:  {left_output_path}")
        print(f"  Right: {right_output_path}")

    print("\n" + "=" * 80)
    print(f"  ✓ Preprocessing complete!")
    print(f"  Left foot shape: {result['left_foot'].shape}, range: [{result['left_foot'].min():.3f}, {result['left_foot'].max():.3f}]")
    print(f"  Right foot shape: {result['right_foot'].shape}, range: [{result['right_foot'].min():.3f}, {result['right_foot'].max():.3f}]")
    print("=" * 80)

    return result


def batch_preprocess_images(input_folder: str, output_folder: str):
    """
    Batch process all foot images in input folder.
    Saves final preprocessed images (left_foot and right_foot) as .npy float32 [0,1].
    """
    image_paths = sorted(glob.glob(os.path.join(input_folder, "*.png")))

    if len(image_paths) == 0:
        print(f"❌ No PNG images found in {input_folder}")
        return {}

    print(f"\n{'='*80}")
    print(f"  BATCH PREPROCESSING - REAL DATASET")
    print(f"{'='*80}")
    print(f"📁 Input folder:  {input_folder}")
    print(f"📂 Output folder: {output_folder}")
    print(f"📊 Found {len(image_paths)} images to process\n")

    results = {}
    successful = 0
    failed = 0

    for idx, img_path in enumerate(image_paths, 1):
        img_name = os.path.basename(img_path)
        print(f"\n[{idx}/{len(image_paths)}] Processing: {img_name}")
        print("-" * 80)

        try:
            result = preprocess_foot_image(
                img_path,
                output_dir=output_folder,
                save_final=True
            )
            if result is not None:
                results[img_name] = result
                successful += 1
                print(f"✓ Successfully processed: {img_name}")
        except Exception as e:
            failed += 1
            print(f"✗ Error processing {img_name}:")
            print(f"  {str(e)}")
            import traceback
            traceback.print_exc()
            results[img_name] = None

    print("\n" + "=" * 80)
    print(f"  BATCH PROCESSING COMPLETE")
    print("=" * 80)
    print(f"✓ Successful: {successful}/{len(image_paths)}")
    print(f"✗ Failed:     {failed}/{len(image_paths)}")
    print(f"\n📂 Output structure:")
    print(f"  {output_folder}/")
    print(f"  ├── left_foot/  (contains name_L.npy files)")
    print(f"  └── right_foot/ (contains name_R.npy files)")
    print("=" * 80)

    return results

## Summary

This notebook provides a complete implementation of the image preprocessing pipeline as described in Chapter 3 of the research methodology. All preprocessing steps plus integration and testing are included.

### Key Outputs:
- ✅ Segmented foot regions
- ✅ Enhanced contrast images
- ✅ Standardized 224×224 RGB images
- ✅ Normalized pixel values [0, 1]

## 9. Test Pipeline with P001.png

**Description:** Test the complete preprocessing pipeline with actual foot image and save outputs from each step.

**Output Location:** `E:\DFU\Image_PRE\` - Contains visualizations from all preprocessing steps.

---

In [ ]:
# ============================================================
# TEST PIPELINE WITH P001.png - COMPLETE WORKFLOW
# ============================================================

test_image_path = r"E:\DFU\Model\img\P001.png"
output_base_dir = r"E:\DFU\Image_PRE"

print("=" * 80)
print("  TESTING COMPLETE PREPROCESSING PIPELINE WITH P001.png")
print("=" * 80)
print(f"\nTest Image: {test_image_path}")
print(f"Output Directory: {output_base_dir}\n")

# ── STEP 1: Image Segmentation (HMRF-EM) ─────────────────────────────────────
print("\n[STEP 1] Image Segmentation (HMRF-EM)...")
img_pil = Image.open(test_image_path).convert("RGB")
image_rgb = np.array(img_pil)
print(f"  Original image shape: {image_rgb.shape}")

image_ydbdr = rgb_to_ydbdr(image_rgb)
labels = hmrf_em_segmentation(image_ydbdr, K=3, beta=1.5)
foot_label = identify_foot_label(labels, image_ydbdr)
foot_mask = create_foot_mask(labels, foot_label)
pure_sole = get_pure_sole_image(image_rgb, foot_mask, background="black")

os.makedirs(output_base_dir, exist_ok=True)
seg_output_path = os.path.join(output_base_dir, "01_segmentation_pure_sole.png")
Image.fromarray(pure_sole).save(seg_output_path)
print(f"  ✓ Saved: {seg_output_path}")

# ── STEP 2: Morphological Dilation ───────────────────────────────────────────
print("\n[STEP 2] Morphological Dilation...")
merged_mask, _ = apply_morphological_dilation(pure_sole, output_dir=output_base_dir)
dilation_output_path = os.path.join(output_base_dir, "02_morphological_dilation.png")
Image.fromarray((merged_mask * 255).astype(np.uint8)).save(dilation_output_path)
print(f"  ✓ Saved: {dilation_output_path}")

# ── STEP 3: L-R Separation & Square Crop ─────────────────────────────────────
print("\n[STEP 3] Left-Right Separation & Square Crop...")
foot_left, foot_right = separate_and_crop_feet(merged_mask, pure_sole, output_dir=output_base_dir)

if foot_left is not None and foot_right is not None:
    Image.fromarray(foot_left).save(os.path.join(output_base_dir, "03_left_foot_square_crop.png"))
    Image.fromarray(foot_right).save(os.path.join(output_base_dir, "03_right_foot_square_crop.png"))
    print(f"  ✓ Saved: 03_left_foot_square_crop.png / 03_right_foot_square_crop.png")
else:
    raise ValueError("Failed to separate feet")

# ── STEP 4: Grayscale & CLAHE Enhancement ────────────────────────────────────
print("\n[STEP 4] Grayscale & CLAHE Enhancement...")
left_gray  = convert_to_grayscale(foot_left)
right_gray = convert_to_grayscale(foot_right)

Image.fromarray(left_gray).save(os.path.join(output_base_dir,  "04a_left_foot_grayscale.png"))
Image.fromarray(right_gray).save(os.path.join(output_base_dir, "04a_right_foot_grayscale.png"))

left_clahe  = apply_clahe(left_gray,  clip_limit=3.5, tile_grid_size=(8, 8))
right_clahe = apply_clahe(right_gray, clip_limit=3.5, tile_grid_size=(8, 8))

Image.fromarray(left_clahe).save(os.path.join(output_base_dir,  "04b_left_foot_clahe.png"))
Image.fromarray(right_clahe).save(os.path.join(output_base_dir, "04b_right_foot_clahe.png"))
print(f"  ✓ Saved: 04a/04b grayscale and CLAHE images")

# ── STEP 5: 3-Channel Conversion ─────────────────────────────────────────────
print("\n[STEP 5] 3-Channel RGB Conversion...")
left_rgb  = convert_to_3channel_rgb(left_clahe)
right_rgb = convert_to_3channel_rgb(right_clahe)
Image.fromarray(left_rgb).save(os.path.join(output_base_dir,  "05_left_foot_3channel_rgb.png"))
Image.fromarray(right_rgb).save(os.path.join(output_base_dir, "05_right_foot_3channel_rgb.png"))
print(f"  ✓ Saved: 05 3-channel RGB images")

# ── STEP 6: Resizing (224×224) ───────────────────────────────────────────────
print("\n[STEP 6] Resizing to 224×224...")
target_size = (224, 224)
left_resized  = resize_image(left_rgb,  target_size=target_size)
right_resized = resize_image(right_rgb, target_size=target_size)
Image.fromarray(left_resized).save(os.path.join(output_base_dir,  "06_left_foot_resized_224x224.png"))
Image.fromarray(right_resized).save(os.path.join(output_base_dir, "06_right_foot_resized_224x224.png"))
print(f"  Shape: {left_resized.shape}  ✓ Saved: 06 resized images")

# ── STEP 7: Pixel Intensity Scaling (÷255) ───────────────────────────────────
print("\n[STEP 7] Pixel Intensity Scaling (÷255) → [0, 1]...")
left_scaled  = scale_pixels(left_resized)
right_scaled = scale_pixels(right_resized)

# Save display version (×255 back to uint8 for PNG)
Image.fromarray((left_scaled  * 255).astype(np.uint8)).save(os.path.join(output_base_dir, "07_left_foot_scaled_display.png"))
Image.fromarray((right_scaled * 255).astype(np.uint8)).save(os.path.join(output_base_dir, "07_right_foot_scaled_display.png"))
print(f"  Left range: [{left_scaled.min():.4f}, {left_scaled.max():.4f}]")
print(f"  Right range: [{right_scaled.min():.4f}, {right_scaled.max():.4f}]")
print(f"  ✓ Saved: 07 scaled display images")

print("\n" + "=" * 80)
print("  ✓ PIPELINE TEST COMPLETE!")
print("=" * 80)
print(f"\nAll outputs saved to: {output_base_dir}")
files = sorted(glob.glob(os.path.join(output_base_dir, "*.png")))
for i, f in enumerate(files, 1):
    size_kb = os.path.getsize(f) / 1024
    print(f"  {i:2d}. {os.path.basename(f):45s} ({size_kb:7.1f} KB)")

## 10. BATCH PROCESSING - Real Images from Dataset

**Description:** the complete preprocessing pipeline with actual foot image in dataset

**Output Location:** `E:\DFU\model\output` - Contains Left and Right with Image Preprocessing

---

In [ ]:
# ============================================================
# BATCH PROCESSING - Real Images from Dataset
# ============================================================

# Define input and output directories
input_images_folder = r"E:\DFU\Model\img"
output_preprocessed_folder = r"E:\DFU\Model\img\output"

# Run batch processing
print("🚀 Starting batch preprocessing of diabetic foot images...\n")
batch_results = batch_preprocess_images(
    input_folder=input_images_folder,
    output_folder=output_preprocessed_folder,
)

# Print summary statistics
print(f"\n📊 FINAL SUMMARY")
print(f"Total images processed: {len(batch_results)}")
successful = sum(1 for v in batch_results.values() if v is not None)
print(f"✓ Successful: {successful}")
print(f"✗ Failed: {len(batch_results) - successful}")

if successful > 0:
    print(f"\n✨ All preprocessed images ready for model training!")
    print(f"📂 Location:")
    print(f"   Left feet:  {output_preprocessed_folder}/left_foot/")
    print(f"   Right feet: {output_preprocessed_folder}/right_foot/")

## 11. CLAHE Parameter Tuning (Dual-Criterion: Entropy + Noise Proxy)

**Goal:** Select the CLAHE clip limit that maximises image entropy (contrast and information content) while remaining below the point at which background noise amplification rises sharply.

**Theoretical basis:**
- **Min et al. (2013):** Image entropy increases monotonically with clip limit but cannot be maximised alone, as very high clip limits amplify noise artefacts.
- **Kuran & Kuran (2021):** A noise proxy (mean absolute difference between the CLAHE output and a Gaussian-blurred version) identifies the threshold at which over-enhancement begins.

**Dual-criterion selection rule:**
1. For each candidate clip limit, compute (a) Shannon entropy of foot pixel intensities within the foot mask and (b) noise proxy = mean absolute difference between the CLAHE image and its Gaussian-blurred counterpart.
2. Select the **smallest clip limit** where entropy is within 1% of its maximum AND the noise proxy has not yet risen sharply (noise proxy ≤ mean + 1 SD across all candidates).

**Tile grid:** Fixed at (8, 8) — eight equal tiles per axis is standard for plantar pressure images of this resolution and is excluded from the sweep.

**When to run:** On the **first 5–10 pilot images from the training set only** (never on held-out test images). The selected clip limit is then fixed uniformly for the entire dataset. Re-determine independently for podoscope data vs. INAOE images.

---


In [ ]:
# ============================================================
# SECTION 11: CLAHE PARAMETER TUNING (Dual-Criterion)
# ============================================================
# Sweep clip limits and select the smallest value where entropy
# is near-maximum AND noise proxy has not risen sharply.
#
# References:
#   Min et al. (2013) — entropy-based CLAHE parameter selection
#   Kuran & Kuran (2021) — noise proxy criterion

import scipy.ndimage as _ndi

# ── Configuration ────────────────────────────────────────────
CLAHE_PILOT_PATH = r"E:\DFU\Model\img\P001.png"   # update to a training-set pilot image
CLIP_CANDIDATES  = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 6.0]
TILE_GRID        = (8, 8)    # fixed — not swept
ENTROPY_TOL      = 0.01      # accept clip limits within 1% of max entropy
NOISE_SD_THRESH  = 1.0       # reject if noise proxy > mean + NOISE_SD_THRESH * std


def compute_image_entropy(image_gray: np.ndarray, foot_mask: np.ndarray | None = None) -> float:
    """
    Shannon entropy of pixel intensity distribution within the foot region.
    Only foot pixels (foot_mask == 1) are used; background is excluded.

    Parameters
    ----------
    image_gray : (H, W) uint8
    foot_mask  : (H, W) binary — 1 = foot pixel. If None, all pixels are used.

    Returns
    -------
    entropy : float (bits)
    """
    if foot_mask is not None:
        pixels = image_gray[foot_mask > 0].ravel()
    else:
        pixels = image_gray.ravel()

    if pixels.size == 0:
        return 0.0

    counts = np.bincount(pixels.astype(np.uint8), minlength=256).astype(np.float64)
    probs  = counts / counts.sum()
    probs  = probs[probs > 0]
    return float(-np.sum(probs * np.log2(probs)))


def compute_noise_proxy(image_clahe: np.ndarray, sigma: float = 3.0) -> float:
    """
    Noise proxy = mean absolute difference between the CLAHE image and a
    Gaussian-blurred version. Captures high-frequency amplification.

    Parameters
    ----------
    image_clahe : (H, W) uint8
    sigma       : Gaussian sigma (default 3.0 px — preserves foot structure, removes noise)

    Returns
    -------
    noise_proxy : float
    """
    blurred = _ndi.gaussian_filter(image_clahe.astype(np.float32), sigma=sigma)
    return float(np.mean(np.abs(image_clahe.astype(np.float32) - blurred)))


# ── Load one pilot image and preprocess up to grayscale ──────
print("Loading pilot image and running preprocessing pipeline...")
_img_rgb   = np.array(Image.open(CLAHE_PILOT_PATH).convert("RGB"))
_ydbdr     = rgb_to_ydbdr(_img_rgb)
_labels    = hmrf_em_segmentation(_ydbdr, K=3, beta=1.5)
_foot_lbl  = identify_foot_label(_labels, _ydbdr)
_foot_mask = create_foot_mask(_labels, _foot_lbl)
_pure      = get_pure_sole_image(_img_rgb, _foot_mask, background="black")

_merged_mask, _ = apply_morphological_dilation(_pure)
_foot_l, _foot_r = separate_and_crop_feet(_merged_mask, _pure)

# Use left foot for the sweep (representative single foot)
_gray = convert_to_grayscale(_foot_l)

# Build a foot-pixel mask on the cropped image (non-zero pixels)
_crop_mask = (_gray > 0).astype(np.uint8)

# ── Run sweep ────────────────────────────────────────────────
print(f"\nSweeping clip_limit over {CLIP_CANDIDATES}, tile_grid={TILE_GRID}\n")

entropies   = []
noise_vals  = []

for clip in CLIP_CANDIDATES:
    enhanced = apply_clahe(_gray, clip_limit=clip, tile_grid_size=TILE_GRID)
    ent  = compute_image_entropy(enhanced, foot_mask=_crop_mask)
    nois = compute_noise_proxy(enhanced)
    entropies.append(ent)
    noise_vals.append(nois)

# ── Print results table ──────────────────────────────────────
print(f"{'clip_limit':>12}  {'entropy (bits)':>16}  {'noise proxy':>12}")
print("-" * 46)
for clip, ent, nois in zip(CLIP_CANDIDATES, entropies, noise_vals):
    print(f"{clip:>12.1f}  {ent:>16.4f}  {nois:>12.4f}")

# ── Apply dual criterion ─────────────────────────────────────
max_entropy  = max(entropies)
noise_mean   = np.mean(noise_vals)
noise_sd     = np.std(noise_vals)
noise_thresh = noise_mean + NOISE_SD_THRESH * noise_sd

print(f"\nMax entropy      : {max_entropy:.4f} bits")
print(f"Noise threshold  : {noise_thresh:.4f}  (mean + {NOISE_SD_THRESH}×SD)")
print(f"Entropy tolerance: within {ENTROPY_TOL*100:.0f}% of max ({max_entropy*(1-ENTROPY_TOL):.4f})")

selected = None
for clip, ent, nois in zip(CLIP_CANDIDATES, entropies, noise_vals):
    entropy_ok = ent >= max_entropy * (1 - ENTROPY_TOL)
    noise_ok   = nois <= noise_thresh
    if entropy_ok and noise_ok:
        selected = clip
        break   # smallest clip limit satisfying both criteria

print("\n--- Recommendation ---")
if selected is not None:
    print(f"✓ Selected clip_limit = {selected}")
    print(f"  Entropy: {entropies[CLIP_CANDIDATES.index(selected)]:.4f} bits  "
          f"(within {ENTROPY_TOL*100:.0f}% of max)")
    print(f"  Noise proxy: {noise_vals[CLIP_CANDIDATES.index(selected)]:.4f}  "
          f"(below threshold {noise_thresh:.4f})")
    print(f"\n  → Update apply_clahe() default and preprocess_foot_image() calls:")
    print(f"      clip_limit={selected}, tile_grid_size=(8, 8)")
    CLAHE_CLIP_LIMIT = selected
else:
    print("✗ No clip limit satisfied both criteria simultaneously.")
    print("  Inspect the table above and choose manually.")
    CLAHE_CLIP_LIMIT = None

print(f"\nCLAHE_CLIP_LIMIT = {CLAHE_CLIP_LIMIT}  (use this in apply_clahe)")


## 12. S2 Dataset Creation (Flip Left Foot)

**Description:** Create S2 dataset from S1 output by horizontally flipping all left foot images.

**Rationale (Chapter 3, Section 3.3):** In S2, the left foot image is flipped horizontally so that the medial arch aligns with the same side as in the right foot image, normalising foot orientation across all training samples for RQ2.

**Steps:**
- Read all `left_foot/P###_L.png` files from S1 output (`batch_preprocess_images` output folder)
- Apply horizontal flip: `img[:, ::-1, :]`
- Copy `right_foot/P###_R.png` files unchanged
- Write to a parallel `output_S2/` directory

**Note:** For ROI annotations on S2 left foot images, x-coordinates must be transformed by `x' = W - x - w_b` (see Section 13).

---

In [ ]:
# ============================================================
# SECTION 12: S2 DATASET CREATION (Flip Left Foot)
# ============================================================

import shutil

def create_s2_dataset(s1_output_folder: str, s2_output_folder: str) -> tuple:
    """
    Create S2 dataset from S1 output by horizontally flipping all left foot images.
    Right foot images are copied unchanged.

    S1 structure (produced by batch_preprocess_images):
        s1_output_folder/
        ├── left_foot/P###_L.png
        └── right_foot/P###_R.png

    S2 structure (left foot horizontally flipped):
        s2_output_folder/
        ├── left_foot/P###_L.png   ← flipped
        └── right_foot/P###_R.png  ← same as S1

    Parameters
    ----------
    s1_output_folder : str
        Path to S1 output (produced by batch_preprocess_images)
    s2_output_folder : str
        Path to write S2 dataset

    Returns
    -------
    (left_s2_dir, right_s2_dir) : tuple of str
    """
    left_s1_dir  = os.path.join(s1_output_folder, 'left_foot')
    right_s1_dir = os.path.join(s1_output_folder, 'right_foot')
    left_s2_dir  = os.path.join(s2_output_folder, 'left_foot')
    right_s2_dir = os.path.join(s2_output_folder, 'right_foot')
    os.makedirs(left_s2_dir,  exist_ok=True)
    os.makedirs(right_s2_dir, exist_ok=True)

    # Left foot: flip horizontally (img[:, ::-1, :])
    left_files = sorted(glob.glob(os.path.join(left_s1_dir, '*.png')))
    print(f"Flipping {len(left_files)} left foot images...")
    for src in left_files:
        fname   = os.path.basename(src)
        img     = np.array(Image.open(src))
        flipped = img[:, ::-1, :]
        Image.fromarray(flipped).save(os.path.join(left_s2_dir, fname))

    # Right foot: copy unchanged
    right_files = sorted(glob.glob(os.path.join(right_s1_dir, '*.png')))
    print(f"Copying  {len(right_files)} right foot images (unchanged)...")
    for src in right_files:
        shutil.copy2(src, os.path.join(right_s2_dir, os.path.basename(src)))

    print(f"\n✓ S2 dataset created:")
    print(f"  Left  (flipped): {left_s2_dir}  ({len(left_files)} files)")
    print(f"  Right (copied):  {right_s2_dir}  ({len(right_files)} files)")
    return left_s2_dir, right_s2_dir


# ── Run S2 creation ──────────────────────────────────────────
s1_folder = r"E:\DFU\Model\img\output"      # S1 from batch_preprocess_images
s2_folder = r"E:\DFU\Model\img\output_S2"   # S2 destination

create_s2_dataset(s1_folder, s2_folder)

## 13. ROI Bounding Box Coordinate Transformation

**Description:** Transform VIA2 bounding box annotations from raw image coordinates to 224×224 model input coordinates, for both S1 and S2 orientations.

**Background (Chapter 3, Section 3.1):** ROI annotations are drawn on raw podoscope images before preprocessing. The bounding box coordinates must be propagated through each preprocessing step to remain correctly aligned with the final model input used in RQ4.

**Transformation steps:**
1. **Tight crop** — subtract crop offset `(x_min, y_min)` of the foot's bounding box
2. **Square padding** — add zero-padding offset `(pad_w_left, pad_h_top)`
3. **Resize to 224×224** — scale coordinates proportionally: `x' = x × (224 / square_size)`
4. **S2 flip (left foot only)** — apply `x' = 224 - x - w_b` (Chapter 3 formula)

**Note:** Segmentation and morphological dilation steps do not alter image dimensions, so raw VIA2 coordinates remain valid through those stages.

---

In [ ]:
# ============================================================
# SECTION 13: ROI BOUNDING BOX COORDINATE TRANSFORMATION
# ============================================================
# Required for RQ4: transform expert ROI annotations (VIA2) to 224×224 model space.

def pad_to_square_with_info(img_array: np.ndarray) -> tuple:
    """
    Identical to pad_to_square() but also returns padding offsets.

    Returns
    -------
    padded        : (size, size, C) zero-padded image
    pad_h_top     : int — rows added above the cropped image
    pad_w_left    : int — columns added to the left of the cropped image
    """
    h, w = img_array.shape[:2]
    size = max(h, w)
    pad_h_top    = (size - h) // 2
    pad_h_bottom = size - h - pad_h_top
    pad_w_left   = (size - w) // 2
    pad_w_right  = size - w - pad_w_left
    padded = np.pad(img_array,
                    ((pad_h_top, pad_h_bottom), (pad_w_left, pad_w_right), (0, 0)),
                    mode='constant', constant_values=0)
    return padded, pad_h_top, pad_w_left


def separate_and_crop_feet_with_info(merged_mask: np.ndarray, img_array: np.ndarray):
    """
    Same as separate_and_crop_feet() but also returns per-foot crop and padding info
    needed to transform ROI bounding box coordinates.

    Returns
    -------
    foot_left_img, foot_right_img : square images (same as separate_and_crop_feet)
    left_info, right_info : dicts with keys
        'x_min', 'y_min'      — tight crop top-left corner in raw image space
        'crop_w', 'crop_h'    — tight crop dimensions before padding
        'pad_h_top', 'pad_w_left' — zero-padding offsets added for square
        'square_size'         — final square side length (px)
    """
    structure = np.ones((3, 3), dtype=np.int32)
    labeled_mask, num_features = label(merged_mask, structure=structure)
    if num_features < 2:
        print("Error: fewer than 2 components found.")
        return None, None, None, None

    component_sizes = np.bincount(labeled_mask.ravel())
    component_sizes[0] = 0
    largest_labels = np.argsort(component_sizes)[-2:]
    slices = find_objects(labeled_mask)

    extracted = []
    for lbl in largest_labels:
        bbox = slices[lbl - 1]
        y_sl, x_sl = bbox
        x_min, y_min = x_sl.start, y_sl.start
        cropped = img_array[y_sl, x_sl]
        crop_h, crop_w = cropped.shape[:2]
        square, pad_h_top, pad_w_left = pad_to_square_with_info(cropped)
        extracted.append({
            'center_x':   (x_sl.start + x_sl.stop) / 2.0,
            'square_img': square,
            'info': {
                'x_min': x_min, 'y_min': y_min,
                'crop_w': crop_w, 'crop_h': crop_h,
                'pad_h_top': pad_h_top, 'pad_w_left': pad_w_left,
                'square_size': square.shape[0],
            },
        })

    extracted.sort(key=lambda d: d['center_x'])
    return (extracted[0]['square_img'], extracted[1]['square_img'],
            extracted[0]['info'],       extracted[1]['info'])


def transform_roi_to_model_input(roi_boxes: list, foot_info: dict,
                                  target_size: int = 224) -> list:
    """
    Transform VIA2 bounding boxes from raw image coordinates to 224×224 model space.

    Transformation (Chapter 3, Section 3.1):
      Step 1: subtract tight crop offset  (x_min, y_min)
      Step 2: add square-padding offset   (pad_w_left, pad_h_top)
      Step 3: scale proportionally        to target_size

    Parameters
    ----------
    roi_boxes : list of dicts — each with keys 'x', 'y', 'width', 'height', 'foot' ('L'/'R')
    foot_info : dict from separate_and_crop_feet_with_info()
    target_size : int, default 224

    Returns
    -------
    transformed : list of dicts with same keys, coordinates in model input space
    """
    scale = target_size / foot_info['square_size']
    transformed = []
    for box in roi_boxes:
        x1 = (box['x'] - foot_info['x_min'] + foot_info['pad_w_left']) * scale
        y1 = (box['y'] - foot_info['y_min'] + foot_info['pad_h_top'])  * scale
        transformed.append({
            'x':      round(x1),
            'y':      round(y1),
            'width':  round(box['width']  * scale),
            'height': round(box['height'] * scale),
            'foot':   box.get('foot', ''),
        })
    return transformed


def transform_roi_s2_flip(roi_boxes_model: list, target_size: int = 224) -> list:
    """
    Apply S2 horizontal flip to already-transformed left foot ROI coordinates.
    Formula (Chapter 3): x' = W - x - w_b

    Parameters
    ----------
    roi_boxes_model : list from transform_roi_to_model_input() (left foot only)
    target_size : int — image width W after resize (default 224)

    Returns
    -------
    flipped : list of dicts with mirrored x coordinates
    """
    W = target_size
    return [{
        'x':      W - box['x'] - box['width'],
        'y':      box['y'],
        'width':  box['width'],
        'height': box['height'],
        'foot':   box.get('foot', ''),
    } for box in roi_boxes_model]


# ── Usage example ────────────────────────────────────────────────────────────
# Step 1: run pipeline with info variant (instead of separate_and_crop_feet)
# foot_left, foot_right, left_info, right_info = separate_and_crop_feet_with_info(merged_mask, pure_sole)
#
# Step 2: load VIA2 export (list of dicts with x, y, width, height, foot)
# raw_roi_left  = [{'x': 80, 'y': 120, 'width': 40, 'height': 30, 'foot': 'L'}]
# raw_roi_right = [{'x': 200, 'y': 150, 'width': 50, 'height': 35, 'foot': 'R'}]
#
# Step 3: transform to 224×224 space (S1)
# roi_left_s1  = transform_roi_to_model_input(raw_roi_left,  left_info,  target_size=224)
# roi_right_s1 = transform_roi_to_model_input(raw_roi_right, right_info, target_size=224)
#
# Step 4: apply S2 flip to left foot coords only (right foot unchanged)
# roi_left_s2  = transform_roi_s2_flip(roi_left_s1, target_size=224)
# roi_right_s2 = roi_right_s1

print("✓ ROI coordinate transformation functions defined.")
print("  pad_to_square_with_info(img_array)")
print("  separate_and_crop_feet_with_info(merged_mask, img_array)")
print("  transform_roi_to_model_input(roi_boxes, foot_info, target_size=224)")
print("  transform_roi_s2_flip(roi_boxes_model, target_size=224)")

## 14. Dilation Kernel Parameter Sweep

**Goal:** Find the minimum morphological dilation kernel size (as % of image height) that reliably connects all toe segments to the main foot region without merging the left and right feet.

**Why this matters:** The kernel percentage in `apply_morphological_dilation()` must be justified empirically. Too small → toes remain disconnected and `separate_and_crop_feet()` fails or produces fragmented crops. Too large → left and right feet merge into one component.

**Success criterion per image:**
- Exactly **2 valid connected components** after dilation (one per foot)
- Each component covers ≥ 5 % of the image area (filters small phantom blobs)
- `separate_and_crop_feet()` returns two non-None crops

**When to run:** On the **first 5–10 pilot images** from the actual hospital dataset, before running `batch_preprocess_images()` on the full collection. The selected `DILATION_PCT` value then replaces the hard-coded `0.12` in `apply_morphological_dilation()`.

---


In [ ]:
# ============================================================
# SECTION 14: DILATION KERNEL PARAMETER SWEEP
# ============================================================
# Run this on the first 5-10 pilot images from the hospital before
# running batch_preprocess_images() on the full dataset.
#
# After running: set DILATION_PCT to the recommended value and update
# the hard-coded 0.12 in apply_morphological_dilation().

from scipy.ndimage import label as ndi_label
import pandas as pd

# ── Configuration ────────────────────────────────────────────
PILOT_DIR  = r"E:\DFU\Model\img"   # ← update to folder with first pilot images
PCT_CANDIDATES = [0.04, 0.06, 0.08, 0.10, 0.12, 0.15, 0.18, 0.20]
MIN_AREA_PCT   = 0.05              # component must cover ≥ 5% of image to count as a foot


def _dilate_with_pct(pure_sole: np.ndarray, pct: float) -> np.ndarray:
    """Apply morphological dilation with kernel = pct * image_height."""
    H = pure_sole.shape[0]
    kernel_px = max(3, int(H * pct))
    kernel = np.ones((kernel_px, 5), dtype=bool)
    gray   = pure_sole.sum(axis=2)
    binary = (gray > 0).astype(np.uint8)
    return binary_dilation(binary, structure=kernel).astype(np.uint8)


def _evaluate_dilation(dilated: np.ndarray) -> dict:
    """
    Check success criteria on a dilated binary mask.
    Returns dict with n_valid_components and status string.
    """
    H, W        = dilated.shape
    min_area    = MIN_AREA_PCT * H * W
    labeled, n  = ndi_label(dilated)
    sizes       = [np.sum(labeled == i) for i in range(1, n + 1)]
    valid       = [s for s in sizes if s >= min_area]
    n_valid     = len(valid)

    if n_valid == 2:
        status = "OK"
    elif n_valid == 1:
        status = "merged"   # left+right merged — kernel too large
    elif n_valid == 0:
        status = "failed"   # nothing survived — image issue
    else:
        status = f">{n_valid} regions"  # toes not fully connected

    return {"n_valid": n_valid, "status": status}


# ── Load pilot images ────────────────────────────────────────
import glob as _glob
pilot_paths = sorted(_glob.glob(os.path.join(PILOT_DIR, "*.png")))[:10]

if not pilot_paths:
    print("⚠  No images found in PILOT_DIR. Update the path and re-run.")
else:
    rows = []

    for img_path in pilot_paths:
        img_name  = os.path.basename(img_path)
        image_rgb = np.array(Image.open(img_path).convert("RGB"))

        # Segment once per image
        image_ydbdr = rgb_to_ydbdr(image_rgb)
        labels_seg  = hmrf_em_segmentation(image_ydbdr, K=3, beta=1.5)
        foot_label  = identify_foot_label(labels_seg, image_ydbdr)
        foot_mask   = create_foot_mask(labels_seg, foot_label)
        pure_sole   = get_pure_sole_image(image_rgb, foot_mask, background="black")

        row = {"image": img_name}
        for pct in PCT_CANDIDATES:
            dilated = _dilate_with_pct(pure_sole, pct)
            ev      = _evaluate_dilation(dilated)
            row[f"{int(pct*100)}%"] = ev["status"]
        rows.append(row)

    # ── Print results table ──────────────────────────────────
    col_keys = [f"{int(p*100)}%" for p in PCT_CANDIDATES]
    header   = f"{'Image':<20}" + "".join(f"{c:>10}" for c in col_keys)
    print(header)
    print("-" * len(header))
    for r in rows:
        line = f"{r['image']:<20}" + "".join(f"{r[c]:>10}" for c in col_keys)
        print(line)

    # ── Recommendation ───────────────────────────────────────
    print("\n--- Recommendation ---")
    passing = [p for p in PCT_CANDIDATES
               if all(rows[i][f"{int(p*100)}%"] == "OK" for i in range(len(rows)))]

    if passing:
        best_pct = min(passing)
        best_px  = int(best_pct * np.array(Image.open(pilot_paths[0])).shape[0])
        print(f"✓ Minimum kernel that passes ALL images: {int(best_pct*100)}% of image height")
        print(f"  (= {best_px} px for {os.path.basename(pilot_paths[0])})")
        print(f"\n  → Update apply_morphological_dilation():")
        print(f"      dilate_h = max(10, int(img_h * {best_pct}))")
        DILATION_PCT = best_pct
    else:
        print("✗ No single kernel size passed all images.")
        print("  Check images where status != OK and inspect manually.")
        DILATION_PCT = None

    print(f"\nDILATION_PCT = {DILATION_PCT}  (use this in apply_morphological_dilation)")
